In [1]:
import sys
print(">>>", sys.executable, "<<<")
from geopy.geocoders import Nominatim
import pandas as pd
import time
import openrouteservice
import numpy as np

>>> c:\Users\vmlle\AppData\Local\Programs\Python\Python311\python.exe <<<


In [2]:
# API do ORS: https://openrouteservice.org/
# Para gerar chave da API: https://account.heigit.org/manage/key

In [3]:
caminho_geojson = r"C:\Users\vmlle\OneDrive\Documentos\Data Science - USP\Modelos otimizacao\geoportal_distrito_municipal_v2 (1).geojson"
gdf = gpd.read_file(caminho_geojson)

print(gdf.columns)
print(gdf.head(3))

NameError: name 'gpd' is not defined

In [ ]:
caminho_geojson = r"C:\Users\vmlle\OneDrive\Documentos\Data Science - USP\Modelos otimizacao\geoportal_distrito_municipal_v2 (1).geojson"
gdf = gpd.read_file(caminho_geojson)

# Garantir WGS84
if gdf.crs is None or gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)

# Centroides
gdf["centroid"] = gdf.geometry.centroid
gdf["latitude"] = gdf["centroid"].y
gdf["longitude"] = gdf["centroid"].x

# DataFrame de distritos
df = gdf[["nm_distrito_municipal", "latitude", "longitude"]].rename(
    columns={"nm_distrito_municipal": "distrito"}
)
df.head()

C:\Users\vmlle\AppData\Local\Temp\ipykernel_20760\4222245894.py:12: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["centroid"] = gdf.geometry.centroid


,distrito,latitude,longitude
0,RIO PEQUENO,-23.571233,-46.758935
1,CIDADE TIRADENTES,-23.584802,-46.400847
2,JARDIM SAO LUIS,-23.680730,-46.739401
3,PONTE RASA,-23.515672,-46.493732
4,LIBERDADE,-23.566955,-46.631957


In [ ]:
ordem = pd.Categorical(df["distrito"], categories=distritos_sp, ordered=True)
df = df.sort_values("distrito", key=lambda s: ordem).reset_index(drop=True)

In [ ]:
# Montar matriz de distâncias com ORS

import numpy as np
import openrouteservice

client = openrouteservice.Client(
    key="5b3ce3597851110001cf62480fa1a8e2672b4020922a679483fb1a60"
)

coords = list(zip(df["longitude"], df["latitude"]))  # (lon, lat)
distritos = df["distrito"].tolist()

n = len(coords)
matriz = np.zeros((n, n))
BLOCO = 50  # mantém requests pequenas

for i in range(0, n, BLOCO):
    for j in range(0, n, BLOCO):
        coords_origem = coords[i:i+BLOCO]
        coords_destino = coords[j:j+BLOCO]

        locations = coords_origem + coords_destino

        sources = list(range(len(coords_origem)))
        destinations = list(range(len(coords_origem), len(locations)))

        matrix = client.distance_matrix(
            locations=locations,
            sources=sources,
            destinations=destinations,
            profile="foot-walking",
            metrics=["distance"],
            units="m"
        )

        dist = np.array(matrix["distances"])
        matriz[i:i+len(coords_origem), j:j+len(coords_destino)] = dist

matriz_df = pd.DataFrame(matriz, index=distritos, columns=distritos)

In [47]:
# salvar matriz para usar no modelo
caminho_csv = r"C:\Users\vmlle\OneDrive\Documentos\Data Science - USP\Modelos otimizacao\matriz_distancias_distritos.csv"
matriz_df.to_csv(caminho_csv, index=True)

In [48]:
matriz_df

,RIO PEQUENO,CIDADE TIRADENTES,JARDIM SAO LUIS,PONTE RASA,LIBERDADE,VILA PRUDENTE,CACHOEIRINHA,SAO LUCAS,CAMBUCI,VILA MARIA,...,JARDIM ANGELA,BRAS,PARQUE DO CARMO,JACANA,SOCORRO,PARI,MARSILAC,SE,VILA MEDEIROS,TUCURUVI
RIO PEQUENO,0.00,47296.43,16479.48,33802.84,17246.67,24333.36,20173.23,28228.36,18910.07,24873.15,...,22653.04,18265.55,39000.81,27309.20,18581.38,19259.93,56698.98,16666.16,25594.55,23389.57
CIDADE TIRADENTES,47296.43,0.00,47542.72,16803.04,31215.66,23035.08,40035.39,18784.12,29112.12,26580.99,...,54639.75,29165.80,9158.19,31785.31,46002.96,29563.87,76842.99,30921.77,28932.31,32206.66
JARDIM SAO LUIS,16479.48,47542.72,0.00,39348.70,20994.53,26423.09,33133.61,30318.09,22254.34,30785.85,...,8308.54,23676.42,41341.73,34907.83,5821.99,25294.42,43939.60,22267.22,32217.91,31803.98
PONTE RASA,33802.84,16803.04,39348.70,0.00,18558.58,15369.33,24792.59,12404.95,16970.20,11338.20,...,46761.58,15672.21,9461.94,16542.52,37577.35,16070.27,72864.66,17428.18,13689.52,16963.87
LIBERDADE,17246.67,31215.66,20994.53,18558.58,0.00,8918.23,16520.96,12813.24,2172.02,10699.18,...,28091.56,3920.37,23139.02,15465.93,19223.18,5852.52,56250.04,2522.13,12776.00,12343.46
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
PARI,19259.93,29563.87,25294.42,16070.27,5852.52,10374.66,12575.27,14039.78,5399.11,5792.88,...,32707.30,2223.05,21268.24,9914.86,23523.07,0.00,61527.29,3286.47,7224.93,7561.17
MARSILAC,56698.98,76842.99,43939.60,72864.66,56250.04,58588.38,71120.48,60123.26,57220.89,66542.84,...,42486.30,59824.13,70860.38,71140.70,38117.61,61527.29,0.00,58446.77,68450.77,68018.23
SE,16666.16,30921.77,22267.22,17428.18,2522.13,9660.44,14086.54,13325.57,2767.92,8777.91,...,29680.10,1866.66,22626.15,12899.89,20495.87,3286.47,58446.77,0.00,10209.96,9777.41
VILA MEDEIROS,25594.55,28932.31,32217.91,13689.52,12776.00,15784.22,12722.79,16599.33,12113.96,4221.84,...,39630.78,8937.90,20676.74,3750.77,30446.55,7224.93,68450.77,10209.96,0.00,4363.61


In [50]:
matriz_df.loc["SANTANA"].sort_values().head(10)

SANTANA              0.00
VILA GUILHERME    3674.77
CASA VERDE        3968.60
BOM RETIRO        4315.96
PARI              4487.10
TUCURUVI          4892.48
MANDAQUI          6007.90
SANTA CECILIA     6091.98
SE                6176.27
LIMAO             6202.59
Name: SANTANA, dtype: float64